# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Árboles: no necesitan escalado, solo imputación (robusta a outliers)
tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), feature_columns),
])

# Modelos basados en distancias / coeficientes: imputación + estandarización
scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
])

preprocessors = {"tree": tree_preprocessor, "scaled": scaled_preprocessor}
preprocessors

{'tree': ColumnTransformer(transformers=[('numeric', SimpleImputer(strategy='median'),
                                  ['alcohol', 'malic_acid', 'ash',
                                   'alcalinity_of_ash', 'magnesium',
                                   'total_phenols', 'flavanoids',
                                   'nonflavanoid_phenols', 'proanthocyanins',
                                   'color_intensity', 'hue',
                                   'od280/od315_of_diluted_wines', 'proline'])]),
 'scaled': ColumnTransformer(transformers=[('numeric',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['alcohol', 'malic_acid', 'ash',
                                   'alcalinity_of_ash', 'magnesium',
                                   'total_phenols', 'flavanoids',
   

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si el `StandardScaler` se ajusta con **todo** el conjunto de entrenamiento antes de la validación cruzada, la media y la desviación estándar que usa incluyen los ejemplos del fold de validación. Así, información de los datos "no vistos" se filtra al entrenamiento (**data leakage**) y la estimación de CV queda optimista. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` vuelven a ajustar el escalador **solo con los folds de entrenamiento** de cada partición y luego lo aplican al fold de validación, que es exactamente lo que pasará en producción con datos nuevos. Además, el pipeline guardado incluye el escalador ya ajustado, de modo que la inferencia aplica la misma transformación automáticamente.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocessing", clone(preprocessors["tree"])),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessing", clone(preprocessors["scaled"])),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessing", clone(preprocessors["scaled"])),
        ("classifier", KNeighborsClassifier()),
    ]),
}
list(models)

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
        "gap": scores["train_score"].mean() - scores["test_score"].mean(),
    })

initial_comparison = (
    pd.DataFrame(rows)
    .sort_values("val_accuracy", ascending=False)
    .reset_index(drop=True)
)
initial_comparison.round(4)

,model,train_accuracy,val_accuracy,val_std,gap
0,logistic_regression,1.0000,0.9791,0.0277,0.0209
1,knn,0.9771,0.9507,0.0353,0.0264
2,decision_tree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__criterion": ["gini", "entropy"],
        "classifier__max_depth": [2, 3, 4, 5, None],
        "classifier__min_samples_leaf": [1, 3, 5],
    },
    "logistic_regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11, 15],
        "classifier__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []

for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline,
        param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append({
        "model": name,
        "cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "best_params": json.dumps(search.best_params_),
    })

tuned_comparison = (
    pd.DataFrame(rows)
    .sort_values(["cv_accuracy", "cv_std"], ascending=[False, True])
    .reset_index(drop=True)
)
tuned_comparison.round(4)

,model,cv_accuracy,cv_std,train_accuracy,best_params
0,logistic_regression,0.9791,0.0277,1.0000,"{""classifier__C"": 1}"
1,knn,0.9579,0.0405,0.9648,"{""classifier__n_neighbors"": 7, ""classifier__we..."
2,decision_tree,0.9022,0.0586,0.9982,"{""classifier__criterion"": ""gini"", ""classifier_..."


## 6. Selección y persistencia del campeón

In [8]:
# Mayor accuracy de CV; en caso de empate, menor desviación estándar
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_

cv_results = initial_comparison.merge(
    tuned_comparison, on="model", suffixes=("_default", "_tuned")
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

best_index = searches[champion_name].best_index_
training_metadata = {
    "champion": champion_name,
    "selection_metric": "accuracy",
    "cv_folds": 5,
    "cv_score": float(searches[champion_name].best_score_),
    "cv_std": float(searches[champion_name].cv_results_["std_test_score"][best_index]),
    "best_params": searches[champion_name].best_params_,
    "features": contract["features"],
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)
print("Modelo campeón:", champion_name)
print("Accuracy CV:", round(training_metadata["cv_score"], 4), "±", round(training_metadata["cv_std"], 4))
print("Hiperparámetros:", training_metadata["best_params"])

Modelo campeón: logistic_regression
Accuracy CV: 0.9791 ± 0.0277
Hiperparámetros: {'classifier__C': 1}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

**Campeón: Regresión Logística (`C = 1`, con imputación y estandarización).**

| Modelo | Accuracy CV (tuned) | Desv. estándar | Accuracy train |
|---|---|---|---|
| Regresión Logística (`C=1`) | **0.979** | **0.028** | 1.000 |
| KNN (`k=7`, uniform) | 0.958 | 0.041 | 0.965 |
| Árbol de decisión (gini, `max_depth=4`) | 0.902 | 0.059 | 0.998 |

- **Desempeño:** tiene el mayor accuracy de validación cruzada (≈ 97.9 %), unos 2 puntos por encima de KNN y 8 por encima del árbol.
- **Variabilidad:** tiene la menor desviación estándar entre folds (≈ 0.028), así que su resultado es el más estable. El árbol es el más inestable (≈ 0.059).
- **Sobreajuste:** su accuracy de train es 1.0, pero la brecha con validación es pequeña (≈ 2 puntos). El árbol también llega a ~1.0 en train, pero con una brecha de ≈ 10 puntos, lo que es un sobreajuste claro. KNN tiene la brecha más pequeña, pero rinde menos.
- **Complejidad:** es un modelo lineal con 3 × 13 coeficientes y un único hiperparámetro. Es rápido de entrenar y de predecir y, a diferencia de KNN, no necesita guardar todo el conjunto de entrenamiento.
- **Interpretabilidad:** los coeficientes, sobre variables estandarizadas, indican cuánto empuja cada variable hacia cada clase, y además entrega probabilidades calibradas razonablemente. El árbol sería más fácil de leer como reglas, pero su peor desempeño y su inestabilidad no compensan esa ventaja.

Las tres familias lineal / distancias / árboles se compararon con el mismo `StratifiedKFold(5)`, y la decisión se tomó **sin usar el test**.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.